In [1]:
from pathlib import Path
from dataclasses import dataclass
import pynucastro as pyna

In [2]:



@dataclass(frozen=True)
class Entry:
    key: tuple        # dedupe key
    sort_key: tuple   # (Zp, Np)
    src: int          # 1 or 2
    idx: int          # order within its file among extracted entries
    block: tuple      # (l1, l2, l3)


def _is_chapter_header(line: str) -> bool:
    return bool(line) and line[0].isdigit()


def _looks_like_rate_line(line: str) -> bool:
    # robust for cases like 'he3    t' (padding spaces)
    if len(line) < 15:
        return False
    parent = line[5:10].strip()
    daughter = line[10:15].strip()
    return bool(parent) and bool(daughter)


def _parse_parent_daughter(line1: str):
    parent = line1[5:10].strip()
    daughter = line1[10:15].strip()
    p = pyna.nucdata.nucleus.Nucleus(parent)
    d = pyna.nucdata.nucleus.Nucleus(daughter)
    return p, d


def _read_chapters_with_blocks(path: Path, block_len: int = 3):
    chapters_order = []
    chapter_items = {}
    current = None

    lines = path.read_text().splitlines(True)

    i = 0
    while i < len(lines):
        line = lines[i]

        if _is_chapter_header(line):
            current = int(line[0])
            if current not in chapter_items:
                chapter_items[current] = []
                chapters_order.append(current)
            chapter_items[current].append(line)
            i += 1
            continue

        if current is not None and _looks_like_rate_line(line) and i + (block_len - 1) < len(lines):
            chapter_items[current].append(tuple(lines[i:i + block_len]))
            i += block_len
            continue

        chapter_items.setdefault(current, []).append(line)
        i += 1

    return chapters_order, chapter_items


def _extract_chapter1_rate_entries(chapter_items, src: int):
    """
    Extract ALL 3-line rate blocks from chapter 1 (weak rates),
    not only beta-.
    """
    entries = []
    other_items = []
    k = 0

    for item in chapter_items:
        if isinstance(item, tuple) and len(item) == 3 and _looks_like_rate_line(item[0]):
            l1 = item[0]
            p, d = _parse_parent_daughter(l1)

            # Dedupe key: parent+daughter + reaction label field (to avoid collisions)
            # reaction label is usually around cols 43:47 in many reaclibs; keep it conservative:
            
            key = (p.Z, p.N, d.Z, d.N)

            entries.append(Entry(key=key, sort_key=(p.Z, p.N), src=src, idx=k, block=item))
            k += 1
        else:
            other_items.append(item)

    return entries, other_items


def _find_insertion_index(chapter_items):
    # after header + immediate spacer lines
    hdr = None
    for i, it in enumerate(chapter_items):
        if isinstance(it, str) and _is_chapter_header(it):
            hdr = i
            break
    if hdr is None:
        return 0

    j = hdr + 1
    while j < len(chapter_items):
        it = chapter_items[j]
        if isinstance(it, str):
            if it.strip() == "":
                j += 1
                continue
            if len(it.rstrip("\n")) == 74 and it.rstrip("\n").isspace():
                j += 1
                continue
        break
    return j


def merge_weak_chapter1(input_file1, input_file2, output_file, block_len: int = 3):
    """
    Chapter 1:
      - keep ALL rates from file1
      - add rates from file2 that are not in file1
      - sort by parent (Z,N); stable within file1 and file2 for ties
    Other chapters: copied from file1 unchanged.
    """
    input_file1 = Path(input_file1)
    input_file2 = Path(input_file2)
    output_file = Path(output_file)

    order1, chap1 = _read_chapters_with_blocks(input_file1, block_len=block_len)
    _,     chap2 = _read_chapters_with_blocks(input_file2, block_len=block_len)

    ch = 1
    items1 = chap1.get(ch, [])
    items2 = chap2.get(ch, [])

    e1, non_rate_items1 = _extract_chapter1_rate_entries(items1, src=1)
    e2, _               = _extract_chapter1_rate_entries(items2, src=2)

    keys1 = {e.key for e in e1}
    e2_new = [e for e in e2 if e.key not in keys1]

    merged = e1 + e2_new
    merged.sort(key=lambda e: (e.sort_key[0], e.sort_key[1], e.src, e.idx))

    merged_blocks = [e.block for e in merged]

    ins = _find_insertion_index(non_rate_items1)
    chap1[ch] = non_rate_items1[:ins] + merged_blocks + non_rate_items1[ins:]

    with output_file.open("w") as fout:
        for chapter in order1:
            for item in chap1.get(chapter, []):
                if isinstance(item, tuple):
                    for ln in item:
                        fout.write(ln)
                else:
                    fout.write(item)



In [3]:
merge_weak_chapter1('Reaclib_exp_R1','Tian_beta_R1','Exp_Tian_beta_R1')